# Shared router fine-tuning

Prepare the catalog-conditioned training bundle for Clef-Flash, check the trainer configuration and stage the Azure upload. Default cells are offline: no downloads, training, Azure submission, or test-set scoring. Labels remain AI-derived silver. See [the guide](../docs/data-and-training.md#shared-router-fine-tuning-data-and-clef-plan) for the recipe, costs and evaluation limits.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

ROOT = Path.cwd()
if not (ROOT / 'pyproject.toml').is_file():
    ROOT = ROOT.parent
if not (ROOT / 'routing_lab').is_dir():
    raise RuntimeError('Open this notebook from the project or notebooks directory')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
SOURCE = ROOT / 'data' / 'public-routing' / 'gpt56-sol-labels-coding-v1-split-20261002'
BUNDLE = ROOT / 'data' / 'finetuning' / 'shared-router-v3'
CONFIG = ROOT / 'azure' / 'training' / 'config.json'
SMOKE_CONFIG = ROOT / 'azure' / 'training' / 'smoke-config.json'
STAGED = ROOT / 'outputs' / 'clef-smoke-upload'

## Freeze the training-derived partitions

Preparation targets 80/10/10 inside train90, preserving group boundaries. The original test10 rows are not exported into the training package. Existing bundles are verified, never overwritten. The test set has already informed configuration choices; it is not a pristine final holdout.

In [ ]:
from routing_lab.finetune_data import verify_bundle

if not BUNDLE.exists():
    subprocess.run([sys.executable, '-m', 'routing_lab.finetune_data',
                    '--source', str(SOURCE), '--output', str(BUNDLE)],
                   cwd=ROOT, check=True)
manifest = verify_bundle(BUNDLE)
print(json.dumps(manifest, indent=2))


In [ ]:
# Verifies the bundle and configuration and prints the planned step count. It does not import Torch,
# load weights, or call Azure.
subprocess.run([sys.executable, '-m', 'routing_lab.finetune', '--bundle', str(BUNDLE),
                '--config', str(CONFIG), '--dry-run'], cwd=ROOT, check=True)

## Stage the smoke job

Enable this to create a small allowlisted upload directory containing code, the verified training bundle and the smoke configuration. It never contains test data, and it does not submit a job or create a GPU. Submitting needs the explicit consent flags shown in the guide, and bills an H100.

In [ ]:
STAGE_AZURE = False
if STAGE_AZURE:
    subprocess.run([sys.executable, '-m', 'azure.training.stage', 'train', '--bundle', str(BUNDLE),
                    '--config', str(SMOKE_CONFIG), '--output', str(STAGED)], cwd=ROOT, check=True)
else:
    print('Azure staging disabled. No upload or paid job started.')

## Inspect a completed run

After a job finishes, download its `results` output into a folder and point `RUN` at it. Compare the per-epoch validation macro F1 for both tracks, the fitted temperature and the reload check in `run.json`. Test-set scoring is a separate, explicitly approved step and is not part of this notebook.

In [ ]:
RUN = ROOT / 'outputs' / 'clef-train-results' / 'run'
report_path = RUN / 'run.json'
if report_path.exists():
    report = json.loads(report_path.read_text(encoding='utf-8'))
    print(report['status'], '| best epoch:', report.get('best_epoch'), '| temperature:', report.get('calibration', {}).get('temperature'))
    for epoch in report['epochs']:
        print(epoch['epoch'], round(epoch['mean_training_loss'], 4), epoch['validation_macro_f1'])
    print('reload check:', report.get('round_trip'))
else:
    print('No training run downloaded yet. Preparation and training are separate steps.')